# Historical Pool Data — FinTech 590
Fetches daily historical TVL, APY, and impermanent loss for each pool in `data/top_pools.csv`.

**Run `defi_pipeline.ipynb` first** to generate the pool list.

Data source: [DeFiLlama Chart API](https://yields.llama.fi/docs) — free, no API key required.  
Coverage: daily data from **March 2022 → present** (~1,400+ data points per pool).

## 0. Imports

In [1]:
import time, pathlib
import requests
import pandas as pd

POOLS_CSV    = pathlib.Path("data/top_pools.csv")
HISTORY_CSV  = pathlib.Path("data/pool_history.csv")
CHART_URL    = "https://yields.llama.fi/chart/"
DELAY        = 0.5   # be polite to the API

print("Ready.")

Ready.


## 1. Load Pool List

In [ ]:
pools = pd.read_csv(POOLS_CSV)

# If llama_id column is missing (old CSV), fetch UUIDs from DeFiLlama by matching address
if "llama_id" not in pools.columns:
    print("llama_id column missing — fetching from DeFiLlama...")
    from web3 import Web3
    FACTORY        = "0x1F98431c8aD98523631AE4a59f267346ea31F984"
    INIT_CODE_HASH = "e34f199b19b2b4f47f68442619d555527d244f78a3297ea89325f843f87b8b54"

    def compute_addr(a, b, fee):
        t0, t1 = sorted([a.lower(), b.lower()])
        enc  = bytes.fromhex(t0[2:]).rjust(32, b"\x00") + bytes.fromhex(t1[2:]).rjust(32, b"\x00") + fee.to_bytes(32, "big")
        salt = Web3.keccak(enc)
        data = b"\xff" + bytes.fromhex(FACTORY[2:]) + salt + bytes.fromhex(INIT_CODE_HASH)
        return Web3.to_checksum_address("0x" + Web3.keccak(data)[12:].hex())

    resp = requests.get("https://yields.llama.fi/pools", timeout=30)
    addr_to_uuid = {}
    for p in resp.json()["data"]:
        if p.get("project") != "uniswap-v3" or p.get("chain") != "Ethereum":
            continue
        tokens = p.get("underlyingTokens") or []
        try:
            fee = int(float(p.get("poolMeta", "0").replace("%", "")) * 10_000)
            if len(tokens) >= 2 and fee > 0:
                addr_to_uuid[compute_addr(tokens[0], tokens[1], fee)] = p["pool"]
        except Exception:
            pass

    pools["llama_id"] = pools["address"].map(addr_to_uuid)
    missing = pools["llama_id"].isna().sum()
    print(f"  Matched {len(pools) - missing}/{len(pools)} pools" + (f" ({missing} skipped)" if missing else ""))

print(f"Loaded {len(pools)} pools from {POOLS_CSV}")
pools[["address", "token0", "token1", "fee_tier", "llama_id"]]

## 2. Fetch Historical Data

For each pool, calls `yields.llama.fi/chart/{llama_id}` which returns daily snapshots with:

| Field | Description |
|-------|-------------|
| `tvl_usd` | Total value locked (USD) |
| `apy` | Total APY (fees + rewards) |
| `apy_base` | APY from trading fees only |
| `apy_base_7d` | 7-day rolling average base APY |
| `il_7d` | 7-day impermanent loss (%) |

In [ ]:
all_records = []

for _, pool in pools.iterrows():
    if pd.isna(pool.get("llama_id")):
        print(f"  Skipping {pool['token0']}/{pool['token1']} — no llama_id")
        continue

    label = f"{pool['token0']}/{pool['token1']} {pool['fee_tier']/1e4:.2f}%"
    print(f"  Fetching {label} ...", end="", flush=True)

    try:
        r = requests.get(CHART_URL + pool["llama_id"], timeout=30)
        r.raise_for_status()
        data = r.json().get("data", [])

        for point in data:
            all_records.append({
                "address":     pool["address"],
                "token0":      pool["token0"],
                "token1":      pool["token1"],
                "fee_tier":    pool["fee_tier"],
                "date":        point["timestamp"][:10],
                "tvl_usd":     point.get("tvlUsd"),
                "apy":         point.get("apy"),
                "apy_base":    point.get("apyBase"),
                "apy_base_7d": point.get("apyBase7d"),
                "il_7d":       point.get("il7d"),
            })

        print(f"  {len(data)} days")

    except Exception as e:
        print(f"  ERROR: {e}")

    time.sleep(DELAY)

print(f"\nTotal records collected: {len(all_records):,}")

## 3. Save to CSV

In [ ]:
if not all_records:
    raise RuntimeError("No records collected — check that llama_id values are present and the API is reachable.")

df = pd.DataFrame(all_records)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["address", "date"]).reset_index(drop=True)

df.to_csv(HISTORY_CSV, index=False)

print(f"Saved {len(df):,} rows to {HISTORY_CSV}")
print(f"Date range : {df['date'].min().date()}  →  {df['date'].max().date()}")
print(f"Pools      : {df['address'].nunique()}")
print()
df.head(10)

## 4. Quick Sanity Checks

In [ ]:
# Records per pool
print("Days of history per pool:")
summary = (
    df.groupby(["token0", "token1", "fee_tier"])
    .agg(days=("date", "count"), earliest=("date", "min"), latest=("date", "max"))
    .reset_index()
)
summary["fee_pct"] = summary["fee_tier"] / 1e4
summary.drop(columns="fee_tier", inplace=True)
print(summary.to_string(index=False))

In [ ]:
# TVL over time for top 5 pools by current TVL
import matplotlib.pyplot as plt

top5 = pools.head(5)["address"].tolist()
fig, ax = plt.subplots(figsize=(12, 5))

for addr in top5:
    subset = df[df["address"] == addr]
    row    = pools[pools["address"] == addr].iloc[0]
    label  = f"{row['token0']}/{row['token1']} {row['fee_tier']/1e4:.2f}%"
    ax.plot(subset["date"], subset["tvl_usd"] / 1e6, label=label)

ax.set_title("TVL Over Time — Top 5 Uniswap V3 Pools")
ax.set_ylabel("TVL (USD millions)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("data/tvl_history.png", dpi=150)
plt.show()
print("Chart saved to data/tvl_history.png")